# EDA: Adult Income Census

In [ ]:
import sys
import os

sys.path.append(os.path.join(os.getcwd(), "../.."))

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency

## Load Dataset

In [ ]:
df = pd.read_csv(
    "hf://datasets/scikit-learn/adult-census-income/adult.csv", na_values=["?"]
)

X = df.drop(columns=["income"]).copy()
y = df[["income"]].copy()

print(f"Features : {X.shape}")
print(f"Targets  : {y.shape}")

## Initial Exploration

In [ ]:
X.head()

In [ ]:
X.info()

In [ ]:
y.head()

| Column          | Type        | Description                          |
|-----------------|-------------|--------------------------------------|
| age             | Continuous  | Age of the individual                |
| workclass       | Categorical | Employment sector                    |
| fnlwgt          | Continuous  | Final weight (census)                |
| education       | Categorical | Education level (text)               |
| education.num   | Ordinal     | Education level (numeric)            |
| marital.status  | Categorical | Marital status                       |
| occupation      | Categorical | Type of occupation                   |
| relationship    | Categorical | Relationship status                  |
| race            | Categorical | Race/Ethnicity                       |
| sex             | Binary      | Male or Female                       |
| capital.gain    | Continuous  | Capital gains                        |
| capital.loss    | Continuous  | Capital losses                       |
| hours.per.week  | Continuous  | Hours worked per week                |
| native.country  | Categorical | Country of origin                    |
| income          | Target      | Binary: <=50K or >50K                |

### Missing Values

In [ ]:
missing = X.isna().sum()
print(missing[missing > 0] if missing.any() else "No missing values in features.")
print()
print("Target missing:", y.isna().sum().item())

## Univariate Analysis

### Numerical Features

In [ ]:
num_features = [
    "age",
    "fnlwgt",
    "education.num",
    "capital.gain",
    "capital.loss",
    "hours.per.week",
]


def univariate_num(df, feature):
    summary = df[feature].describe()
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.histplot(data=df, x=feature, kde=True, ax=axes[0])
    axes[0].set_title(f"Distribution of {feature}")
    sns.boxplot(x=df[feature], ax=axes[1])
    axes[1].set_title(f"Boxplot of {feature}")
    plt.tight_layout()
    plt.show()
    return summary


for feat in num_features:
    print(f"── {feat} ──")
    print(univariate_num(X, feat))
    print()

#### Target: `income`

In [ ]:
income_counts = y["income"].value_counts()
print("Income distribution:")
print(income_counts)
print()
print("Percentages:")
print(y["income"].value_counts(normalize=True) * 100)

### Categorical Features

In [ ]:
cat_features = [
    "workclass",
    "education",
    "marital.status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "native.country",
]


def univariate_cat(df, feature):
    counts = df[feature].value_counts()
    percentages = df[feature].value_counts(normalize=True) * 100
    summary = pd.DataFrame({"Count": counts, "Percentage (%)": percentages})
    plt.figure(figsize=(8, 4))
    sns.countplot(data=df, x=feature, order=counts.index)
    plt.title(f"Distribution of {feature}")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()
    return summary


for feat in cat_features:
    print(f"── {feat} ──")
    print(univariate_cat(X, feat))
    print()

## Bivariate Analysis

### Numerical Features × Target

In [ ]:
def bivariate_num(X, y, feature):
    data = pd.concat([X[[feature]], y], axis=1)
    data.columns = [feature, "income"]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.boxplot(data=data, x="income", y=feature, ax=axes[0])
    axes[0].set_title(f"{feature} by income")
    sns.violinplot(data=data, x="income", y=feature, ax=axes[1])
    axes[1].set_title(f"Violin: {feature} by income")
    plt.tight_layout()
    plt.show()


for feat in num_features:
    print(f"── {feat} ──")
    bivariate_num(X, y, feat)

### Categorical Features × Target

In [ ]:
def bivariate_cat(X, y, feature):
    data = pd.concat([X[[feature]], y], axis=1)
    data.columns = [feature, "income"]

    # Count by category and income
    crosstab = pd.crosstab(data[feature], data["income"])

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    # Stacked bar plot
    crosstab.plot(kind="bar", ax=axes[0])
    axes[0].set_title(f"Count by {feature} and income")
    axes[0].set_xlabel(feature)
    axes[0].set_ylabel("Count")
    axes[0].legend(title="income")
    axes[0].tick_params(axis="x", rotation=45)

    # Normalized bar plot (percentages)
    crosstab_norm = crosstab.div(crosstab.sum(axis=1), axis=0) * 100
    crosstab_norm.plot(kind="bar", stacked=True, ax=axes[1])
    axes[1].set_title(f"Percentage by {feature} and income")
    axes[1].set_xlabel(feature)
    axes[1].set_ylabel("Percentage (%)")
    axes[1].legend(title="income")
    axes[1].tick_params(axis="x", rotation=45)

    plt.tight_layout()
    plt.show()


for feat in cat_features:
    print(f"── {feat} ──")
    bivariate_cat(X, y, feat)

### Correlation Heatmap (Numerical Features)

In [ ]:
corr = X[num_features].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr, cmap="coolwarm", annot=True, fmt=".2f")
plt.title("Pearson Correlation Matrix (Numerical Features)")
plt.tight_layout()
plt.show()

**Observations:**
- `fnlwgt` is a census weight with minimal distribution variance — likely low predictive value
- `capital.gain` and `capital.loss` are sparse (many zeros), indicate special circumstances
- `age`, `education.num`, and `hours.per.week` show reasonable distributions

### Feature Associations (Categorical)

#### `education` vs `education.num`

In [ ]:
edu_check = (
    X[["education", "education.num"]].drop_duplicates().sort_values("education.num")
)
print(edu_check)
print()
print(
    "Conclusion: education.num is ordinal encoding of education. Drop education, keep education.num"
)

#### `marital.status` vs `relationship`

In [ ]:
tab = pd.crosstab(X["marital.status"], X["relationship"])
chi2, p, dof, _ = chi2_contingency(tab)
n = tab.sum().sum()
cramers_v = np.sqrt((chi2 / n) / (min(tab.shape) - 1))
print(f"Cramér's V (marital.status × relationship): {cramers_v:.3f}  (p={p:.2e})")
print()
print(
    "Conclusion: Very strong association (V ≈ 0.97). Keep relationship (more granular), drop marital.status"
)

## Preprocessing Decisions

| Decision | Reason |
|---|---|
| Drop `education` | Redundant with `education.num` (ordinal encoding) |
| Drop `marital.status` | Redundant with `relationship` (Cramér's V = 0.97) |
| Drop `fnlwgt` | Census weight with minimal variance, low predictive value |
| Handle missing values | Fill `occupation`, `workclass`, `native.country` with mode |
| Consolidate `workclass` | Merge rare/similar categories (e.g., all government types) |
| Consolidate `race` | Merge rare categories → 'Other' |
| Consolidate `native.country` | Merge non-US → 'Other' |
| Encode `sex` | Binary: Female=1, Male=0 |
| Encode `income` (target) | Binary: >50K=1, <=50K=0 |
| One-hot encode categorical | `workclass`, `occupation`, `relationship`, `race`, `native.country` |

# EDA: Adult Income Census

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency

## Load Dataset

In [ ]:
df = pd.read_csv(
    "hf://datasets/scikit-learn/adult-census-income/adult.csv", na_values=["?"]
)
print(f"Dataset shape: {df.shape}")

## Initial Exploration

In [ ]:
df.head()

In [ ]:
df.info()

## Missing Values Analysis

In [ ]:
missing_pct = df.isna().sum() / df.shape[0] * 100
print("Missing values percentage:")
print(missing_pct[missing_pct > 0])

**Observation:** Very small percentage of missing values in categorical features (`occupation`, `workclass`, `native.country`) — can be filled with mode.

## Feature Cardinality

In [ ]:
df.nunique()

## Categorical Features Analysis

### Redundant Features: `education` vs `education.num`

In [ ]:
df[["education", "education.num"]].drop_duplicates().sort_values("education.num")

**Observation:** `education.num` is an ordinal encoding of `education`. We can drop `education` and keep the numeric version.

### Association: `marital.status` vs `relationship`

In [ ]:
tab = pd.crosstab(df["marital.status"], df["relationship"])
chi2, p, dof, expected = chi2_contingency(tab)
print(f"Chi-squared test statistic: {chi2:.2f}")
print(f"p-value: {p:.4f}")

# Calculate Cramér's V
n = tab.sum().sum()
phi2 = chi2 / n
r, k = tab.shape
cramers_v = np.sqrt(phi2 / min(k - 1, r - 1))
print(f"Cramér's V: {cramers_v:.3f}")

**Observation:** Very strong association (V ≈ 0.97). We should keep only one: `relationship` is more granular than `marital.status`.

### Binary Features: `sex` and `income`

In [ ]:
print("sex distribution:")
print(df["sex"].value_counts())
print()
print("income distribution (target):")
print(df["income"].value_counts())

**Observation:** Binary features — can be encoded as 0/1. Income is imbalanced (24% >50K, 76% <=50K).

###`workclass` Distribution

In [ ]:
df["workclass"].value_counts()

**Observation:** Can consolidate:
- 'State-gov', 'Local-gov', 'Federal-gov' → 'Government'
- 'Self-emp-not-inc', 'Self-emp-inc' → 'Self-employed'
- 'Without-pay', 'Never-worked' → 'Other'

### `occupation` Distribution

In [ ]:
df["occupation"].value_counts()

### `race` Distribution

In [ ]:
df["race"].value_counts()

**Observation:** Can consolidate rare categories: 'Asian-Pac-Islander', 'Amer-Indian-Eskimo' → 'Other'

### `native.country` Distribution

In [ ]:
print(f"Unique countries: {df['native.country'].nunique()}")
print(f"US: {(df['native.country'] == 'United-States').sum()}")
print(f"Non-US: {(df['native.country'] != 'United-States').sum()}")

**Observation:** Mostly 'United-States'. Consolidate non-US → 'Other'

## Numerical Features Analysis

In [ ]:
num_features = [
    "age",
    "fnlwgt",
    "education.num",
    "capital.gain",
    "capital.loss",
    "hours.per.week",
]
df[num_features].describe()

### Correlation Analysis

In [ ]:
corr_matrix = df[num_features].corr()
sns.heatmap(corr_matrix, cmap="coolwarm", annot=True, fmt=".2f")
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()

**Observation:** No strong linear correlations.

### Distributions

In [ ]:
for num_feature in num_features:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.histplot(df, x=num_feature, ax=axes[0])
    sns.boxplot(df[num_feature], ax=axes[1])
    axes[0].set_title(f"Distribution of {num_feature}")
    axes[1].set_title(f"Boxplot of {num_feature}")
    plt.tight_layout()
    plt.show()